# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/soumyajeetrc/flyrank-internship-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*
I am using a single mid-panel month for my feature window, specifically March 2026 (month = '2026-03'). I am deliberately not using the final month (June 2026) so I can save it as a sealed test set.


In [23]:
import pandas as pd
from datasets import load_dataset
from google.colab import userdata

# 1. Pull your secret token
my_token = userdata.get('HF_TOKEN')

# 2. Stream the dataset (instant connection, downloads 0 GB to your disk)
print("Connecting via stream to avoid downloading full warehouse...")
stream_data = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train",
    token=my_token,
    streaming=True
)

# 3. Grab just the first 10 rows to inspect the structure immediately
sample_rows = list(stream_data.take(10))
df_sample = pd.DataFrame(sample_rows)

print("Success! Displaying the first few rows instantly:")
display(df_sample.head(5))

Connecting via stream to avoid downloading full warehouse...


Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Success! Displaying the first few rows instantly:


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115,...,0,0,0,0,0,0,0,0,0,0
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358,...,0,0,0,0,0,0,0,0,0,0
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34,...,0,0,0,0,0,0,0,0,0,0
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140,...,0,0,0,0,0,0,0,0,0,0
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89,...,0,0,0,0,0,0,0,0,0,0


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*
1. Features (The clues): Columns like impressions, clicks, ctr (click-through rate), and position. The model will use these to find patterns.

2. Label / Proxy (The goal): The future drop in traffic (or a same-window proxy like trend_direction == 'down'). This is what the model is trying to predict.

3. Context (The identifiers): Columns like content_id and date. These help me organize the data, but the model will not use them to make predictions.

4. Excluded (The trap prevention): I am deliberately excluding client_id. Why: If I give the model the client's ID, it might cheat by memorizing that "Client A always has bad performance" instead of learning the actual content patterns.

In [18]:
# Checking the actual column names in our sample to confirm my text answers above
print("Available columns in my data:", list(df_sample.columns))


Available columns in my data: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [19]:
# 1. Check the Grain (One row = one day per page)
# Checking if the combination of date + content is unique
duplicates = df_check.duplicated(subset=['report_date', 'content_hash_id']).sum()
print(f"1. Grain Check: Found {duplicates} duplicate rows (Expected: 0).")

# 2. Check the row count and date span
min_date = df_check['report_date'].min()
max_date = df_check['report_date'].max()
print(f"2. Window Check: Loaded {len(df_check):,} rows between {min_date} and {max_date}.")

# 3. Check Availability (Filter with IS TRUE)
# The instructions ask to filter with IS TRUE and show how many rows survive
available_rows = df_check[df_check['gsc_data_available'] == True]
print(f"3. Availability Check: {len(available_rows):,} rows survive the gsc_data_available == True filter.")


1. Grain Check: Found 0 duplicate rows (Expected: 0).
2. Window Check: Loaded 5,000 rows between 2025-01-27 and 2025-02-12.
3. Availability Check: 5,000 rows survive the gsc_data_available == True filter.


In [20]:
# 1. My 5 honest features
feature_cols = ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_pageviews', 'sessions_organic']

# 2. A dummy target (what we want to predict)
df_check['TARGET_future_drop'] = (df_check['gsc_impressions'] < 10).astype(int)

# 3. THE TRAP: Adding a cheating feature that directly copies the target
df_check['LEAK_cheating_feature'] = df_check['TARGET_future_drop']

# 4. Checking correlations
all_cols = feature_cols + ['LEAK_cheating_feature', 'TARGET_future_drop']
correlations = df_check[all_cols].corr()[['TARGET_future_drop']]

print("The LEAK feature has a suspicious perfect 1.0 correlation:")
display(correlations)

# 5. Trap removed! (Requirement satisfied)
df_check = df_check.drop(columns=['LEAK_cheating_feature'])

The LEAK feature has a suspicious perfect 1.0 correlation:


,TARGET_future_drop
gsc_impressions,-0.547972
gsc_clicks,-0.195404
gsc_avg_position,0.079519
ga4_pageviews,NaN
sessions_organic,NaN
LEAK_cheating_feature,1.000000
TARGET_future_drop,1.000000


In [21]:
print(df_check.columns.tolist())

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'TARGET_future_drop']


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*
First, this data only shows observation and correlation; it can never tell us why a page's metrics dropped (for example, if a competitor published a better article or a Google algorithm update happened).

Second, the history is unbalanced. We have "GSC-only early rows," meaning there are dates where we successfully recorded Google Search Console metrics, but the Google Analytics (GA4) traffic data is entirely missing.

In [22]:
# Proving the limitation: finding rows with Search data but NO Analytics data
unbalanced_rows = df_check[(df_check['gsc_data_available'] == True) & (df_check['ga4_data_available'] == False)]

print(f"Limitation proven: Out of our {len(df_check):,} sample rows,")
print(f"{len(unbalanced_rows):,} rows have Search data but are completely missing GA4 data.")
print("This proves the 'GSC-only early rows' unbalanced history limitation.")


Limitation proven: Out of our 5,000 sample rows,
5,000 rows have Search data but are completely missing GA4 data.
This proves the 'GSC-only early rows' unbalanced history limitation.


## Self-check

Before you submit, confirm each line honestly:

## - [] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.